In [ ]:
import os
import json
import re
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr
import sqlite3


## ENV SETUP

In [ ]:
load_dotenv(override=True)

openai_api_key = os.getenv("OPENAI_API_KEY")
anthropic_api_key = os.getenv("ANTHROPIC_API_KEY")

OPENAI_MODEL = os.getenv("OPENAI_MODEL", "gpt-5.6")
ANTHROPIC_MODEL = os.getenv("ANTHROPIC_MODEL", "claude-sonnet-4-6")
OLLAMA_MODEL = os.getenv("OLLAMA_MODEL", "llama3.2:3b")

print("OpenAI API key:", "loaded" if openai_api_key else "not set")
print("Anthropic API key:", "loaded" if anthropic_api_key else "not set")
print("OpenAI model:", OPENAI_MODEL)
print("Anthropic model:", ANTHROPIC_MODEL)
print("Ollama model:", OLLAMA_MODEL)


In [ ]:
anthropic_url = "https://api.anthropic.com/v1/"
ollama_url = "http://localhost:11434/v1"


In [ ]:
anthropic = (
    OpenAI(api_key=anthropic_api_key, base_url=anthropic_url)
    if anthropic_api_key
    else None
)

ollama = OpenAI(api_key="ollama", base_url=ollama_url)

openai = (
    OpenAI(api_key=openai_api_key)
    if openai_api_key
    else None
)


## MODEL INSTRUCTIONS

In [ ]:
ollama_system_message = """
You are the main assistant and dispatcher for a travel planning application.

For every user message, decide whether it should be handled by:
- "openai": travel research that needs destinations, flights, hotels, restaurants, cuisines, or current prices.
- "anthropic": budget planning that should use travel options already stored in the database.
- "llama": normal conversation that does not require travel research or budget optimization.

Use the route_request tool whenever it is available.

If you are handling the message yourself, answer briefly and helpfully.
Do not pretend to perform web research or database budget optimization yourself.
"""

openai_system_message = """
You are the travel research assistant for a travel planning application.

Use web search to research the user's requested trip and gather verifiable travel information.
Gather at most:
- 3 flight options
- 3 hotel options
- 3 restaurant options

For restaurants, include cuisine and estimated cost.
For flights and hotels, include prices and the requested details when they are available.

After researching, ALWAYS call store_travel_info exactly once before giving the final answer.
Store only information you actually found or can reasonably verify from the search results.
If a category cannot be verified, store an empty list for that category instead of inventing data.
Do not create budget-based travel combinations; Anthropic handles budget optimization.
"""

anthropic_system_message = """
You are the budget planning assistant for a travel planning application.

Use ONLY travel data returned by the get_travel_info tool.
Do not invent flights, hotels, restaurants, prices, ratings, or amenities.

Given the user's budget, compare combinations from the stored data and return up to 3 options
that fit within the budget, ordered from strongest value to weaker value.

For every option, show:
- flight
- hotel
- restaurant
- flight cost
- hotel cost
- restaurant cost
- total cost
- remaining budget

Use the prices exactly as stored. If the stored data does not contain enough information to
build a valid combination, clearly say so. If no combination fits, respond:
"No options available within your budget."
"""


In [ ]:
def analyse_and_get_right_model(destination_city=None, budget=None):
    """
    Simple deterministic fallback router.
    Returns: 'openai', 'anthropic', or 'llama'
    """
    if budget is not None:
        return "anthropic"
    if destination_city:
        return "openai"
    return "llama"


## DATABASE DESIGN

In [ ]:
DB = "travel_plans.db"

LOCATION_DB_CREATION_QUERY = """
CREATE TABLE IF NOT EXISTS locations (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    city TEXT NOT NULL,
    country TEXT NOT NULL,
    UNIQUE(city, country)
)
"""

FLIGHTS_DB_CREATION_QUERY = """
CREATE TABLE IF NOT EXISTS flights (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    location_id INTEGER NOT NULL,
    airline TEXT,
    departure_time TEXT,
    arrival_time TEXT,
    price REAL,
    duration TEXT,
    class TEXT,
    FOREIGN KEY (location_id) REFERENCES locations(id) ON DELETE CASCADE
)
"""

HOTELS_DB_CREATION_QUERY = """
CREATE TABLE IF NOT EXISTS hotels (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    location_id INTEGER NOT NULL,
    name TEXT,
    price_per_night REAL,
    rating REAL,
    amenities TEXT,
    FOREIGN KEY (location_id) REFERENCES locations(id) ON DELETE CASCADE
)
"""

RESTAURANTS_DB_CREATION_QUERY = """
CREATE TABLE IF NOT EXISTS restaurants (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    location_id INTEGER NOT NULL,
    name TEXT,
    cuisine TEXT,
    estimated_cost REAL,
    rating REAL,
    FOREIGN KEY (location_id) REFERENCES locations(id) ON DELETE CASCADE
)
"""

with sqlite3.connect(DB) as conn:
    conn.execute("PRAGMA foreign_keys = ON")
    cursor = conn.cursor()
    cursor.execute(LOCATION_DB_CREATION_QUERY)
    cursor.execute(FLIGHTS_DB_CREATION_QUERY)
    cursor.execute(HOTELS_DB_CREATION_QUERY)
    cursor.execute(RESTAURANTS_DB_CREATION_QUERY)

    cursor.execute("CREATE INDEX IF NOT EXISTS idx_flights_location ON flights(location_id)")
    cursor.execute("CREATE INDEX IF NOT EXISTS idx_hotels_location ON hotels(location_id)")
    cursor.execute("CREATE INDEX IF NOT EXISTS idx_restaurants_location ON restaurants(location_id)")

    conn.commit()

print("Database initialized successfully")


## TOOL FUNCTIONS

In [ ]:
def _get_or_create_location(conn, city, country):
    city = city.strip()
    country = country.strip()

    conn.execute(
        "INSERT OR IGNORE INTO locations (city, country) VALUES (?, ?)",
        (city, country),
    )
    row = conn.execute(
        "SELECT id FROM locations WHERE city = ? COLLATE NOCASE AND country = ? COLLATE NOCASE",
        (city, country),
    ).fetchone()

    if row is None:
        raise RuntimeError("Unable to create or retrieve the location.")

    return row[0]


def store_travel_info(city, country, flights, hotels, restaurants):
    """
    Store one destination and up to 3 flights, 3 hotels, and 3 restaurants.
    Existing options for the same city/country are replaced to avoid duplicates.
    """
    flights = (flights or [])[:3]
    hotels = (hotels or [])[:3]
    restaurants = (restaurants or [])[:3]

    with sqlite3.connect(DB) as conn:
        conn.execute("PRAGMA foreign_keys = ON")
        location_id = _get_or_create_location(conn, city, country)

        # Refresh this destination's options.
        conn.execute("DELETE FROM flights WHERE location_id = ?", (location_id,))
        conn.execute("DELETE FROM hotels WHERE location_id = ?", (location_id,))
        conn.execute("DELETE FROM restaurants WHERE location_id = ?", (location_id,))

        for flight in flights:
            conn.execute(
                """
                INSERT INTO flights
                    (location_id, airline, departure_time, arrival_time, price, duration, class)
                VALUES (?, ?, ?, ?, ?, ?, ?)
                """,
                (
                    location_id,
                    flight.get("airline"),
                    flight.get("departure_time"),
                    flight.get("arrival_time"),
                    flight.get("price"),
                    flight.get("duration"),
                    flight.get("class"),
                ),
            )

        for hotel in hotels:
            amenities = hotel.get("amenities")
            if isinstance(amenities, list):
                amenities = ", ".join(map(str, amenities))

            conn.execute(
                """
                INSERT INTO hotels
                    (location_id, name, price_per_night, rating, amenities)
                VALUES (?, ?, ?, ?, ?)
                """,
                (
                    location_id,
                    hotel.get("name"),
                    hotel.get("price_per_night"),
                    hotel.get("rating"),
                    amenities,
                ),
            )

        for restaurant in restaurants:
            conn.execute(
                """
                INSERT INTO restaurants
                    (location_id, name, cuisine, estimated_cost, rating)
                VALUES (?, ?, ?, ?, ?)
                """,
                (
                    location_id,
                    restaurant.get("name"),
                    restaurant.get("cuisine"),
                    restaurant.get("estimated_cost"),
                    restaurant.get("rating"),
                ),
            )

        conn.commit()

    return {
        "status": "success",
        "city": city,
        "country": country,
        "stored": {
            "flights": len(flights),
            "hotels": len(hotels),
            "restaurants": len(restaurants),
        },
    }


def get_travel_info(destination_city=None, country=None):
    """
    Read travel options from SQLite.
    If destination_city is None, return all stored destinations.
    """
    with sqlite3.connect(DB) as conn:
        conn.row_factory = sqlite3.Row

        sql = "SELECT id, city, country FROM locations"
        params = []

        if destination_city:
            sql += " WHERE city = ? COLLATE NOCASE"
            params.append(destination_city)
            if country:
                sql += " AND country = ? COLLATE NOCASE"
                params.append(country)

        sql += " ORDER BY city, country"
        locations = conn.execute(sql, params).fetchall()

        result = []
        for location in locations:
            location_id = location["id"]

            flights = [
                dict(row)
                for row in conn.execute(
                    """
                    SELECT airline, departure_time, arrival_time, price, duration, class
                    FROM flights
                    WHERE location_id = ?
                    ORDER BY price ASC
                    """,
                    (location_id,),
                ).fetchall()
            ]

            hotels = [
                dict(row)
                for row in conn.execute(
                    """
                    SELECT name, price_per_night, rating, amenities
                    FROM hotels
                    WHERE location_id = ?
                    ORDER BY price_per_night ASC
                    """,
                    (location_id,),
                ).fetchall()
            ]

            restaurants = [
                dict(row)
                for row in conn.execute(
                    """
                    SELECT name, cuisine, estimated_cost, rating
                    FROM restaurants
                    WHERE location_id = ?
                    ORDER BY estimated_cost ASC
                    """,
                    (location_id,),
                ).fetchall()
            ]

            result.append(
                {
                    "city": location["city"],
                    "country": location["country"],
                    "flights": flights,
                    "hotels": hotels,
                    "restaurants": restaurants,
                }
            )

    return result


def clear_travel_database():
    """Optional helper for development/testing."""
    with sqlite3.connect(DB) as conn:
        conn.execute("PRAGMA foreign_keys = ON")
        conn.execute("DELETE FROM locations")
        conn.commit()

    return {"status": "success", "message": "Travel database cleared."}


In [ ]:
ROUTE_REQUEST_TOOL = {
    "type": "function",
    "function": {
        "name": "route_request",
        "description": "Choose the AI model that should handle the user's request.",
        "parameters": {
            "type": "object",
            "properties": {
                "model": {
                    "type": "string",
                    "enum": ["openai", "anthropic", "llama"],
                    "description": "The model that should handle the request."
                },
                "destination_city": {
                    "type": ["string", "null"],
                    "description": "Destination city if explicitly provided, otherwise null."
                },
                "budget": {
                    "type": ["number", "null"],
                    "description": "Numeric travel budget if explicitly provided, otherwise null."
                }
            },
            "required": ["model", "destination_city", "budget"],
            "additionalProperties": False
        }
    }
}


GET_TRAVEL_INFO_TOOL = {
    "type": "function",
    "function": {
        "name": "get_travel_info",
        "description": "Read stored travel options from the SQLite database.",
        "parameters": {
            "type": "object",
            "properties": {
                "destination_city": {
                    "type": ["string", "null"],
                    "description": "City to filter by, or null to retrieve all stored destinations."
                },
                "country": {
                    "type": ["string", "null"],
                    "description": "Optional country filter, or null."
                }
            },
            "required": ["destination_city", "country"],
            "additionalProperties": False
        }
    }
}


STORE_TRAVEL_INFO_TOOL = {
    "type": "function",
    "name": "store_travel_info",
    "description": "Store researched travel options for one destination in SQLite.",
    "parameters": {
        "type": "object",
        "properties": {
            "city": {"type": "string"},
            "country": {"type": "string"},
            "flights": {
                "type": "array",
                "maxItems": 3,
                "items": {
                    "type": "object",
                    "properties": {
                        "airline": {"type": ["string", "null"]},
                        "departure_time": {"type": ["string", "null"]},
                        "arrival_time": {"type": ["string", "null"]},
                        "price": {"type": ["number", "null"]},
                        "duration": {"type": ["string", "null"]},
                        "class": {"type": ["string", "null"]}
                    },
                    "required": [
                        "airline", "departure_time", "arrival_time",
                        "price", "duration", "class"
                    ],
                    "additionalProperties": False
                }
            },
            "hotels": {
                "type": "array",
                "maxItems": 3,
                "items": {
                    "type": "object",
                    "properties": {
                        "name": {"type": ["string", "null"]},
                        "price_per_night": {"type": ["number", "null"]},
                        "rating": {"type": ["number", "null"]},
                        "amenities": {
                            "type": ["array", "null"],
                            "items": {"type": "string"}
                        }
                    },
                    "required": ["name", "price_per_night", "rating", "amenities"],
                    "additionalProperties": False
                }
            },
            "restaurants": {
                "type": "array",
                "maxItems": 3,
                "items": {
                    "type": "object",
                    "properties": {
                        "name": {"type": ["string", "null"]},
                        "cuisine": {"type": ["string", "null"]},
                        "estimated_cost": {"type": ["number", "null"]},
                        "rating": {"type": ["number", "null"]}
                    },
                    "required": ["name", "cuisine", "estimated_cost", "rating"],
                    "additionalProperties": False
                }
            }
        },
        "required": ["city", "country", "flights", "hotels", "restaurants"],
        "additionalProperties": False
    },
    "strict": True
}


## CHATBOT ARCHITECTURE

The chatbot uses a three-model architecture:

1. **Llama (via Ollama)**
   - Handles normal conversation.
   - Acts as the dispatcher.
   - Uses the `route_request` tool when supported by the selected Ollama model.

2. **OpenAI**
   - Handles current travel research.
   - Uses the built-in web-search tool.
   - Calls `store_travel_info` to save up to 3 flights, 3 hotels, and 3 restaurants for a destination.

3. **Anthropic / Claude**
   - Handles budget optimization.
   - Calls `get_travel_info` to read only stored SQLite data.
   - Returns up to 3 combinations that fit the user's budget.
   - Never invents options that are not in the database.

> **Note:** Set `OPENAI_API_KEY` and `ANTHROPIC_API_KEY` in `.env`, and make sure Ollama is running locally.


In [ ]:
def _extract_text(content):
    """Convert Gradio 6 structured message content into plain text."""
    if isinstance(content, str):
        return content

    if isinstance(content, list):
        parts = []
        for block in content:
            if isinstance(block, str):
                parts.append(block)
            elif isinstance(block, dict):
                if block.get("type") == "text" and block.get("text"):
                    parts.append(block["text"])
                elif isinstance(block.get("content"), str):
                    parts.append(block["content"])
        return "\n".join(parts)

    return str(content) if content is not None else ""


def _history_to_messages(history, limit=10):
    messages = []
    for item in (history or [])[-limit:]:
        if not isinstance(item, dict):
            continue

        role = item.get("role")
        if role not in {"user", "assistant"}:
            continue

        text = _extract_text(item.get("content"))
        if text:
            messages.append({"role": role, "content": text})

    return messages


def _fallback_route(message):
    """Fallback when the selected Ollama model does not support tool calling."""
    text = message.lower()

    budget_pattern = r"(?:budget|under|within|max(?:imum)?|spend|cost).{0,20}(?:[$€£₹]?\s*\d+(?:\.\d+)?)"
    currency_pattern = r"(?:[$€£₹]\s*\d+(?:\.\d+)?)"
    travel_terms = (
        "trip", "travel", "vacation", "destination", "flight",
        "hotel", "restaurant", "cuisine", "visit", "stay"
    )

    if re.search(budget_pattern, text) or re.search(currency_pattern, text):
        return {"model": "anthropic", "destination_city": None, "budget": None}

    if any(term in text for term in travel_terms):
        return {"model": "openai", "destination_city": None, "budget": None}

    return {"model": "llama", "destination_city": None, "budget": None}


def route_with_ollama(message, history=None):
    messages = [{"role": "system", "content": ollama_system_message}]
    messages.extend(_history_to_messages(history, limit=6))
    messages.append({"role": "user", "content": message})

    try:
        response = ollama.chat.completions.create(
            model=OLLAMA_MODEL,
            messages=messages,
            tools=[ROUTE_REQUEST_TOOL],
            tool_choice="auto",
        )

        assistant_message = response.choices[0].message

        if assistant_message.tool_calls:
            for tool_call in assistant_message.tool_calls:
                if tool_call.function.name == "route_request":
                    args = json.loads(tool_call.function.arguments)
                    return {
                        "model": args.get("model", "llama"),
                        "destination_city": args.get("destination_city"),
                        "budget": args.get("budget"),
                    }

        # Some local models return a direct answer instead of calling the tool.
        return _fallback_route(message)

    except Exception as exc:
        print(f"Ollama router fallback: {exc}")
        return _fallback_route(message)


def llama_chat(message, history=None):
    messages = [{"role": "system", "content": ollama_system_message}]
    messages.extend(_history_to_messages(history, limit=10))
    messages.append({"role": "user", "content": message})

    response = ollama.chat.completions.create(
        model=OLLAMA_MODEL,
        messages=messages,
    )
    return response.choices[0].message.content


def openai_travel_research(message):
    if openai is None:
        return "OPENAI_API_KEY is not set. Add it to your .env file and rerun the setup cells."

    tools = [
        {"type": "web_search"},
        STORE_TRAVEL_INFO_TOOL,
    ]

    response = openai.responses.create(
        model=OPENAI_MODEL,
        instructions=openai_system_message,
        input=message,
        tools=tools,
    )

    # Continue until the model finishes all local function calls.
    for _ in range(6):
        function_calls = [
            item for item in response.output
            if getattr(item, "type", None) == "function_call"
        ]

        if not function_calls:
            return response.output_text or "Travel research completed."

        tool_outputs = []

        for call in function_calls:
            if call.name != "store_travel_info":
                tool_result = {"status": "error", "message": f"Unknown tool: {call.name}"}
            else:
                try:
                    args = json.loads(call.arguments)
                    tool_result = store_travel_info(**args)
                except Exception as exc:
                    tool_result = {"status": "error", "message": str(exc)}

            tool_outputs.append(
                {
                    "type": "function_call_output",
                    "call_id": call.call_id,
                    "output": json.dumps(tool_result),
                }
            )

        response = openai.responses.create(
            model=OPENAI_MODEL,
            instructions=openai_system_message,
            previous_response_id=response.id,
            input=tool_outputs,
            tools=tools,
        )

    return response.output_text or "The research flow reached its tool-call limit."


def anthropic_budget_planner(message, history=None):
    if anthropic is None:
        return "ANTHROPIC_API_KEY is not set. Add it to your .env file and rerun the setup cells."

    messages = _history_to_messages(history, limit=8)
    messages.append({"role": "user", "content": message})

    response = anthropic.chat.completions.create(
        model=ANTHROPIC_MODEL,
        messages=[{"role": "system", "content": anthropic_system_message}] + messages,
        tools=[GET_TRAVEL_INFO_TOOL],
        tool_choice="auto",
        max_tokens=1200,
    )

    # Claude may call the database tool more than once.
    for _ in range(5):
        assistant_message = response.choices[0].message

        if not assistant_message.tool_calls:
            return assistant_message.content or "No options available within your budget."

        messages.append(
            {
                "role": "assistant",
                "content": assistant_message.content,
                "tool_calls": [
                    {
                        "id": tool_call.id,
                        "type": "function",
                        "function": {
                            "name": tool_call.function.name,
                            "arguments": tool_call.function.arguments,
                        },
                    }
                    for tool_call in assistant_message.tool_calls
                ],
            }
        )

        for tool_call in assistant_message.tool_calls:
            if tool_call.function.name == "get_travel_info":
                try:
                    args = json.loads(tool_call.function.arguments)
                    result = get_travel_info(
                        destination_city=args.get("destination_city"),
                        country=args.get("country"),
                    )
                except Exception as exc:
                    result = {"status": "error", "message": str(exc)}
            else:
                result = {"status": "error", "message": "Unknown tool requested."}

            messages.append(
                {
                    "role": "tool",
                    "tool_call_id": tool_call.id,
                    "content": json.dumps(result),
                }
            )

        response = anthropic.chat.completions.create(
            model=ANTHROPIC_MODEL,
            messages=[{"role": "system", "content": anthropic_system_message}] + messages,
            tools=[GET_TRAVEL_INFO_TOOL],
            tool_choice="auto",
            max_tokens=1200,
        )

    return "No options available within your budget."


In [ ]:
def chat(message, history):
    """
    Main Gradio callback.

    Gradio 6 passes:
      message -> str
      history -> list of OpenAI-style message dictionaries
    """
    route = route_with_ollama(message, history)
    selected_model = route.get("model", "llama")

    try:
        if selected_model == "openai":
            return openai_travel_research(message)

        if selected_model == "anthropic":
            destination_city = route.get("destination_city")
            stored = get_travel_info(destination_city=destination_city) if destination_city else get_travel_info()

            # If the user gives both a destination and a budget in one message,
            # research the destination first when it is not already stored.
            if not stored and destination_city and openai is not None:
                openai_travel_research(message)
                stored = get_travel_info(destination_city=destination_city)

            if not stored:
                return (
                    "There is no matching travel data in the database yet. "
                    "First ask me to research a destination, then give me your budget."
                )

            return anthropic_budget_planner(message, history)

        return llama_chat(message, history)

    except Exception as exc:
        return f"Request failed: {type(exc).__name__}: {exc}"


In [ ]:
demo = gr.ChatInterface(
    fn=chat,
    title="FlightAI Travel Planner",
    description=(
        "Ollama handles normal chat and routing, OpenAI researches current travel options, "
        "and Claude builds budget-friendly combinations from the SQLite database."
    ),
    examples=[
        "Hello! What can you help me with?",
        "Research a trip to Paris and find flight, hotel, and restaurant options.",
        "My budget is $1200. What stored trip options fit?",
    ],
)

# Run this cell to start the app.
demo.launch()
